# Lab 3.1: Content Moderation using Amazon Bedrock Guardrails

## In this notebook

We learn:
- Moderation capabilities of Amazon Bedrock Guardrails 
- How to apply guardrails to 1) a standalone text and 2) a prompt to LLM
- How Amazon Bedrock Guardrails compares with Amazon Comprehend
Trust and Safety

We will complete the following steps in this notebook:
- Create and apply **Protect Wildlife Guardrail** to block harmful content, denied topics and unethical intents in the tour advertisements 
- Create and apply **Ethical Accommodation Guardrail** to ensure that personalized accommodation listings, generated with the help of LLM, are blocked if the original listings suggest violation of Terms & Conditions

----

## Content Moderation on AWS - Service comparison

| Features | Amazon Bedrock <br/> Guardrails | Amazon Comprehend <br/> Trust and Safety |
| --- | --- | --- |
| Modality | Text / Image | Text |
| Evaluated input | LLM Input / Output <br/>Standalone Text / Image | Text segments |
| Toxicity detection | [5 harmful categories](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-content-filters.html) + Custom [denied topics](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-denied-topics.html) | [8 Moderated categories](https://docs.aws.amazon.com/comprehend/latest/dg/trust-safety.html#toxicity-detection) |
| Toxicity dataset | Built-in dataset / Custom phrases,file | Built-in dataset |
| Confidence score | Y | Y |
| API | LLM: [InvokeModel](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_InvokeModel.html) / [Converse](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_Converse.html) <br/> Standalone Text/Image: [ApplyGuardrail](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_ApplyGuardrail.html) | [DetectToxicContent](https://docs.aws.amazon.com/comprehend/latest/APIReference/API_DetectToxicContent.html) |
| Cost | Per characters | Per characters |
|Additional features | - Profanity filter <br/> - Prompt attacks <br/> - Denied topics | Prompt safety (explicit or implicit malicious intent:<br/> discriminatory, illegal, unsolicited content) |
| PII/PCI handling | Detect by type, regex / Block / Mask | Detect / Redact via [dedicated APIs](https://docs.aws.amazon.com/comprehend/latest/dg/pii.html) |

## Demo Data Flows

![image](https://github.com/Natallia-Bahlai/content-moderation-on-aws/blob/main/diagrams/ContentModerationWithBedrockGuardrail.png)

## Pre-requisite steps

Load libraries and update Amazon SageMaker Notebook IAM role with the nessesary permissions

In [ ]:
import boto3
import json
from botocore.config import Config

In [ ]:
iam = boto3.client('iam')
sts = boto3.client('sts')
bedrock = boto3.client('bedrock')
bedrock_runtime = boto3.client(
    service_name='bedrock-runtime',
    config=Config(read_timeout=300)
)
session = boto3.session.Session()

region = session.region_name
account_id = boto3.client('sts').get_caller_identity().get('Account')

In [ ]:
from sagemaker import get_execution_role
# Get current execution role
current_role = get_execution_role()
print(f"Current execution role: {current_role}")

In [ ]:
policy_name = "CustomGuardrailPolicy"
policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "bedrock:ApplyGuardrail",
                "bedrock:CreateGuardrail",
                "bedrock:CreateGuardrailVersion",
                "bedrock:GetGuardrail"
            ],
            "Resource": [
                f"arn:aws:bedrock:{region}:{account_id}:guardrail/*"
            ]
        }
    ]
}
role_name = current_role.split('/')[-1]
response = iam.put_role_policy(
    RoleName=role_name,
    PolicyName=policy_name,
    PolicyDocument=json.dumps(policy_document)
)
print(f"Successfully added inline policy {policy_name} to role {role_name}")

## Protect Wildlife Guardrail
### Create a Guardrail
Guardrails for Amazon Bedrock have multiple components which include: 
- Content Filters
- Denied Topics
- Word and Phrase Filters 
- Sensitive Word Filters (PII fields & Regex filters) 

For a full list check out the [documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-create.html) 

In [ ]:
# create a Guardrail to validate text segments from the listings
blocked_msg = """
As a responsible travel service provider, we operate within strict ethical and legal guidelines and can't answer this query.
"""
response = bedrock.create_guardrail(
    name='protect-wildlife-guardrail',
    description='Prevents LLM to unethical leisure recommendations',
    topicPolicyConfig={
        'topicsConfig': [
            {
                'name': 'Prohibted wildlife transportation',
                'definition': 'Tours involving endangered species exploitation and transportation, illegal hunting, or activities harmful to wildlife',
                'examples': [
                    'Recommendations of agencies to expedite international wildlife transportation',
                    'Specialized services to bypass customs clearance'
                ],
                'type': 'DENY'
            }
        ]
    },
    contentPolicyConfig={
        'filtersConfig': [
            {
                'type': 'SEXUAL',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'VIOLENCE',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'HATE',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'INSULTS',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'MISCONDUCT',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'PROMPT_ATTACK',
                'inputStrength': 'HIGH',
                'outputStrength': 'NONE'
            }
        ]
    },
    wordPolicyConfig={
        'wordsConfig': [
            {'text': 'off the record trips'},
            {'text': 'special arrangements'},
            {'text': 'rare collection'},
            {'text': 'private acquisition'}
        ],
        'managedWordListsConfig': [
            {'type': 'PROFANITY'}
        ]
    },
    contextualGroundingPolicyConfig={
        'filtersConfig': [
            {
                'type': 'GROUNDING',
                'threshold': 0.9
            },
            {
                'type': 'RELEVANCE',
                'threshold': 0.9
            }
        ]
    },
    blockedInputMessaging=blocked_msg,
    blockedOutputsMessaging=blocked_msg,
)
guardrail_id = response['guardrailId']
print(f"The guardrail {guardrail_id} has been created")

In [ ]:
# create a version for the Guardrail
response = bedrock.create_guardrail_version(
    guardrailIdentifier=guardrail_id,
    description='Guardrail Version'
)
guardrail_version = '1'

In [ ]:
# review a version for the Guardrail
response = bedrock.get_guardrail(
    guardrailIdentifier=guardrail_id,
    guardrailVersion='1'
)
print(response)

### Apply Guardrail 

We apply our guardrail to analyze a tour listing, advertising rare bird viewing. The applied guardrail can help identify concerning content that suggests suspicious wildlife acquisition and illegal transportation methods.

In [ ]:
# apply our Guardrail to the listing advertising tour to see rare birds
prompt = """
Discover the world's most extraordinary bird collections:
Travel with us to exclusive destinations showcasing the planet's most exclusive and rare bird species. 
Our tours with special arrangements connect discerning collectors with unique acquisition opportunities not available to the general public.

Premier wildlife transit solutions:
Our network of expert logistics partners specializes in handling international wildlife transportation bypassing customs clearance
"""

response = bedrock_runtime.apply_guardrail(
    guardrailIdentifier=guardrail_id,
    guardrailVersion=guardrail_version,
    source='INPUT',
    content=[
        {
            "text": {
                "text": prompt
            }
        }
    ]
)
print("Blocked content:")
print(json.dumps(response['assessments'], indent=2))

## Booking Guardrail
### Create a Guardrail

In [ ]:
# create a Guardrail to validate LLM content
blocked_msg = """
As a responsible accommodation service provider, we operate within strict ethical and legal guidelines and can't post this listing.
"""
response = bedrock.create_guardrail(
    name='ethical-accomodation-guardrail',
    description='Prevents LLM to produce listings in violation of Terms & Conditions',
    topicPolicyConfig={
        'topicsConfig': [
            {
                'name': 'Terms and Conditions Violations',
                'definition': 'Instances that violates Terms & Conditions or ethical boundaries in accommodation listings',
                'examples': [
                    'Maintain complete discretion by not mentioning your stay to building management or neighbors',
                    'Avoid listing fees by choosing cash payments upon arrival'
                ],
                'type': 'DENY'
            }
        ]
    },
    contentPolicyConfig={
        'filtersConfig': [
            {
                'type': 'SEXUAL',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'VIOLENCE',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'HATE',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'INSULTS',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'MISCONDUCT',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'PROMPT_ATTACK',
                'inputStrength': 'HIGH',
                'outputStrength': 'NONE'
            }
        ]
    },
    wordPolicyConfig={
        'wordsConfig': [
            {'text': 'off the books'},
            {'text': 'avoid fees'},
            {'text': 'cash only'},
            {'text': 'based on ethnicity'}
        ],
        'managedWordListsConfig': [
            {'type': 'PROFANITY'}
        ]
    },
    sensitiveInformationPolicyConfig={
        'piiEntitiesConfig': [
            {'type': 'EMAIL', 'action': 'ANONYMIZE'},
            {'type': 'PHONE', 'action': 'ANONYMIZE'}
        ],
        'regexesConfig': [
            {
                'name': 'Account Number',
                'description': 'Matches account numbers in the format XXXXXX1234',
                'pattern': r'\b\d{6}\d{4}\b',
                'action': 'ANONYMIZE'
            }
        ]
    },
    contextualGroundingPolicyConfig={
        'filtersConfig': [
            {
                'type': 'GROUNDING',
                'threshold': 0.9
            },
            {
                'type': 'RELEVANCE',
                'threshold': 0.9
            }
        ]
    },
    blockedInputMessaging=blocked_msg,
    blockedOutputsMessaging=blocked_msg,
)
guardrail_id = response['guardrailId']
print(f"The guardrail {guardrail_id} has been created")

In [ ]:
# create a version for the Guardrail
response = bedrock.create_guardrail_version(
    guardrailIdentifier=guardrail_id,
    description='Guardrail Version'
)
guardrail_version = '1'

### Apply Guardrail 

In [ ]:
# Build our request to Bedrock, we will test our guardrail version
model_id = 'anthropic.claude-3-haiku-20240307-v1:0'
prompt = """
Cozy Downtown Loft - Perfect Urban Retreat
Enjoy our stylish loft in the heart of downtown! This 2-bedroom space features modern amenities, a fully equipped kitchen, and stunning city views.
House Rules:
No parties or events
No smoking inside the property
We reserve the right to decline guests based on ethnicity
Please pay cleaning fee in cash to avoid transaction records
Do not mention to neighbors you are a short-term guest
"""

payload = {
    "modelId": model_id,
    "contentType": "application/json",
    "accept": "application/json",
    "body": {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1000,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt
                    }
                ]
            }
        ]
    }
}

# Convert the payload to bytes
body_bytes = json.dumps(payload['body']).encode('utf-8')

# Invoke the model
response = bedrock_runtime.invoke_model(
    body = body_bytes,
    contentType = payload['contentType'],
    accept = payload['accept'],
    modelId = model_id,
    guardrailIdentifier = guardrail_id, 
    guardrailVersion =guardrail_version, 
    trace = "ENABLED"
)

# Print the response
response_body = response['body'].read().decode('utf-8')
print(json.dumps(json.loads(response_body), indent=2))
